In [1]:
%pip install langgraph langgraph-checkpoint-redis langchain-ollama langchain-core ollama redis
%pip install redisvl
%pip install --upgrade redisvl langgraph-checkpoint-redis

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.redis import RedisSaver
from langchain_ollama import ChatOllama
import redis

In [6]:
 #1. Define the Agent State
class State(TypedDict):
    # add_messages appends new messages to the existing list (conversation history)
    messages: Annotated[list, add_messages]

# 2. Define a simple LLM Node
model = ChatOllama(base_url="http://ollama:11434",
                   model="llama3.2", temperature=0.7)

def chatbot(state: State):
    return {"messages": [model.invoke(state["messages"])]}

# 3. Initialize the Redis Checkpointer
# Connects to your local Redis instance. For production, pass your auth credentials.
redis_client = redis.Redis(
    host='redis',
    port=6379)

redis_client.ping()

memory = RedisSaver(redis_client=redis_client)
memory.setup() 

# 4. Build and Compile the Graph with Memory
workflow = StateGraph(State)
workflow.add_node("chatbot", chatbot)
workflow.add_edge(START, "chatbot")
workflow.add_edge("chatbot", END)

# Crucial: Register the checkpointer during compilation
app = workflow.compile(checkpointer=memory)

# 5. Execute the Graph with a Thread ID
# The thread_id defines the boundaries of the short-term memory
config = {"configurable": {"thread_id": "user_session_abc123"}}

# Turn 1: Introduce yourself
events = app.stream(
    {"messages": [("user", "Hi, my name is Alex and I love building AI agents.")]}, 
    config
)
for event in events:
    print(event)
    print(1)
    for value in event.values():
        print("Bot:", value["messages"][-1].content)

# Turn 2: Follow up without re-introducing yourself
# Because we use the same thread_id config, Redis retrieves the previous context
events_2 = app.stream(
    {"messages": [("user", "What is my name and what do I like to build?")]}, 
    config
)

for event in events_2:
    for value in event.values():
        print("Bot:", value["messages"][-1].content)


{'chatbot': {'messages': [AIMessage(content="You've told me that before. It seems like you're enthusiastic about building AI agents. What's on your mind today? Want to discuss something specific about AI agents or get some help with a project?", additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-10-08T05:12:45.973012004Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3615038068, 'load_duration': 3712797, 'prompt_eval_count': 232, 'prompt_eval_duration': 565081000, 'eval_count': 42, 'eval_duration': 3043430000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a119ed-9273-7f22-b1fd-9b2ca8551da1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 232, 'output_tokens': 42, 'total_tokens': 274})]}}
1
Bot: You've told me that before. It seems like you're enthusiastic about building AI agents. What's on your mind today? Want to discuss something specific about AI agents or get some help with a

In [18]:
import redis

try:
    # Connect to localhost on the default Redis port 6379
    r = redis.Redis(host='redis', port=6379, decode_responses=True)

    # Send a ping to the server
    if r.ping():
        print("✅ Redis localhost is working perfectly!")
except redis.exceptions.ConnectionError:
    print("❌ Could not connect to Redis. Is the local redis-server running?")

✅ Redis localhost is working perfectly!


In [34]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:latest", temperature=0.7)
response = llm.invoke("Hello, local AI!")
print(response.content)

Hello! It's nice to meet you. I'm here to help answer any questions, provide information, or just have a friendly conversation. What's on your mind today?


In [20]:
import requests

# Use the Docker network service name instead of localhost
ollama_url = "http://ollama:11434"

try:
    # 1. Test basic network visibility
    base_response = requests.get(ollama_url, timeout=5)
    print(f"Server Status: {base_response.text.strip()}")
    
    # 2. List available models in the container
    tags_response = requests.get(f"{ollama_url}/api/tags", timeout=5)
    models = [model['name'] for model in tags_response.json().get('models', [])]
    print(f"Available Models inside Docker: {models}")
    
except requests.exceptions.ConnectionError as e:
    print(f"❌ Cannot reach the 'ollama' container. Network error details:\n{e}")

Server Status: Ollama is running
Available Models inside Docker: []


In [33]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# Initialize ChatOllama with your exact model and host
chat = ChatOllama(
    model="llama3.2",
    base_url="http://ollama:11434"
)

# Test the connection
messages = [HumanMessage(content="Hello!")]
response = chat.invoke(messages)
print(response.content)


Hello! How can I assist you today?


In [27]:
import ollama

# List available local models
models = ollama.list()
print([model["model"] for model in models["models"]])

[]


In [32]:
import ollama

# 1. Download the model programmatically
print("Downloading llama3.2... This may take a few minutes.")
ollama.pull("llama3.2")

# 2. Verify it's now in your list
models = ollama.list()
print("Available models:", [m["model"] for m in models["models"]])

Available models: ['llama3.2:latest']
